In [1]:
'''
здесь использую динамическое разрешение; его параметры подобраны так, чтобы точек было МНОГО

цель данного ноутбука - иллюстрация работы свёртки с динамическим разрешением различных вариантов
'''

'\nздесь использую динамическое разрешение; его параметры подобраны так, чтобы точек было МНОГО\n\nцель данного ноутбука - иллюстрация работы свёртки с динамическим разрешением различных вариантов\n'

In [2]:
from read_complex_matrix import *
import timeit
matr = read_complex_matrix('ref_matrix.txt')

In [3]:
from add_roughness import *
from variables import *
fig = plot_density_profile(transform_array(matr, rough_res, lambda z: z))
fig.show()

In [4]:
%%time
from calculating_dynamic_resolution import *

q = dynamic_mesh_q(kmax, Ndots, gap)

r, r_real, r_img = calculate_matrices_and_reflection(matr, rough_res, kmax, q, Ndots, gap)

Ndots = len(r)

CPU times: total: 31.2 ms
Wall time: 40.8 ms


In [5]:
from plotting import *

# Построение графиков
plot_reflection_vs_wavevector(r, q, Ndots)

In [6]:
np.array(r).shape

(1051,)

In [7]:
from convolution import convolution

r_conv = []
r_conv.append(np.array(r))

for i in [0, 0.2, 0.8]:
    r_conv.append(convolution(np.array(r), kmax, q, Ndots_norm, 10.00*sigma, 1.00*sigma, gap*i))

In [8]:
plot_reflection_vs_wavevector_xn(r_conv, q, Ndots, ['без свёртки', 'без сглаживания', 'со сглаживанием 2%', 'со сглаживанием 8%'])

In [9]:
r_conv = []
r_conv.append(np.array(r))

"""
Если брать слишком низкое разрешение, в учёт начинают входить точки, коэф.отражения для которых больше на несколько порядков (отстоящие от исходной на 3σ влево),
что уводит коэф.отражения в исходной точке вверх на несколько порядков => правая часть графика уходит сильно вверх
"""
for i in [5, 7, 9, 25]:
    r_conv.append(convolution(np.array(r), kmax, q, i, 14*sigma, 14*sigma, gap))

In [10]:
plot_reflection_vs_wavevector_xn(r_conv, q, Ndots, ['без свёртки', '5 точек', '7 точек', '9 точек', '25 точек'])

In [11]:
q_s = np.load('qarr.npy')
r_s10 = np.load('rznach10.npy')
r_s = np.load('rznach.npy')
q = q*1e-10

In [12]:
r_conv = []
r_conv.append(convolution(r_s, kmax, q_s, 25, 10*sigma, 10*sigma, gap))
r_b10 = np.array(r_conv).reshape(1500,)
r_delta = np.abs(r_b10[0:1300] - r_s10[0:1300])
delta_r = np.mean(r_delta/r_b10[0:1300])

import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=q_s,
    y=r_b10,
    mode='lines',
    name='b10',
    marker=dict(size=2.2, opacity=1, color='red')
))

fig.add_trace(go.Scatter(
    x=q_s,
    y=r_s10,
    mode='lines',
    name='s10',
    marker=dict(size=2.2, opacity=1, color='blue')
))

fig.add_trace(go.Scatter(
    x=q_s,
    y=r_s,
    mode='lines',
    name='no_conv',
    marker=dict(size=2.2, opacity=1, color='cyan')
))

fig.update_layout(
    xaxis_title='q',
    yaxis_title='r(q)',
    yaxis=dict(type='log'),  
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    width=1400,  
    height=600  
)

fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)
fig.show()

In [13]:
delta_r

np.float64(0.00658665913514709)